In [3]:
import pandas as pd
from pathlib import Path

# Find the project folder automatically
BASE = Path.cwd()
if not (BASE / "data").exists():
    BASE = BASE.parent
RAW = BASE / "data" / "raw"
CLEAN = BASE / "data" / "cleaned"
CLEAN.mkdir(parents=True, exist_ok=True)
print("Reading from:", RAW)
print("Saving to:   ", CLEAN)

customers   = pd.read_csv(RAW / "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
orders      = pd.read_csv(RAW / "olist_orders_dataset.csv")
items       = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments    = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
products    = pd.read_csv(RAW / "olist_products_dataset.csv")
sellers     = pd.read_csv(RAW / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(RAW / "product_category_name_translation.csv")

print("Loaded all 9 tables")

Reading from: C:\Users\user\cart2insights\data\raw
Saving to:    C:\Users\user\cart2insights\data\cleaned
Loaded all 9 tables


In [4]:
# ---------- ORDERS: convert dates, standardise status ----------
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])
orders["order_status"] = orders["order_status"].str.strip().str.lower()

# ---------- ITEMS: convert date ----------
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])

# ---------- REVIEWS: convert dates, keep one review per order (the latest) ----------
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c])

before = len(reviews)
reviews = (
    reviews.sort_values("review_answer_timestamp")
           .drop_duplicates(subset="order_id", keep="last")
           .drop_duplicates(subset="review_id", keep="last")
)
print("reviews:", before, "->", len(reviews))
print("orders", orders.shape, "| items", items.shape, "| reviews", reviews.shape)

reviews: 99224 -> 98095
orders (99441, 8) | items (112650, 7) | reviews (98095, 7)


In [5]:
# ---------- CUSTOMERS ----------
customers["customer_zip_code_prefix"] = customers["customer_zip_code_prefix"].astype(str).str.zfill(5)
customers["customer_city"]  = customers["customer_city"].str.strip().str.lower()
customers["customer_state"] = customers["customer_state"].str.strip().str.upper()

# ---------- SELLERS ----------
sellers["seller_zip_code_prefix"] = sellers["seller_zip_code_prefix"].astype(str).str.zfill(5)
sellers["seller_city"]  = sellers["seller_city"].str.strip().str.lower()
sellers["seller_state"] = sellers["seller_state"].str.strip().str.upper()

# ---------- GEOLOCATION: collapse to ONE row per zip prefix ----------
geolocation["geolocation_zip_code_prefix"] = (
    geolocation["geolocation_zip_code_prefix"].astype(str).str.zfill(5)
)
geolocation["geolocation_city"]  = geolocation["geolocation_city"].str.strip().str.lower()
geolocation["geolocation_state"] = geolocation["geolocation_state"].str.strip().str.upper()

geolocation = (
    geolocation.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean"),
        geolocation_city=("geolocation_city", lambda s: s.mode().iloc[0]),
        geolocation_state=("geolocation_state", lambda s: s.mode().iloc[0]),
    )
)

# ---------- PAYMENTS: remove invalid rows, installments minimum 1 ----------
before = len(payments)
payments = payments[
    (payments["payment_type"] != "not_defined") & (payments["payment_value"] > 0)
].copy()
payments["payment_installments"] = payments["payment_installments"].clip(lower=1)
print("payments:", before, "->", len(payments))
print("geolocation", geolocation.shape)

payments: 103886 -> 103877
geolocation (19015, 5)


In [6]:
# ---------- Fix the misspelled column names ----------
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

# ---------- Add the 2 categories missing from the translation table ----------
extra = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_food_preparers"],
})
category_translation = (
    pd.concat([category_translation, extra], ignore_index=True)
      .drop_duplicates(subset="product_category_name")
      .reset_index(drop=True)
)

# ---------- Missing category -> 'unknown' (also add it to the translation table) ----------
products["product_category_name"] = products["product_category_name"].fillna("unknown")
unknown_row = pd.DataFrame({"product_category_name": ["unknown"],
                            "product_category_name_english": ["unknown"]})
category_translation = pd.concat([category_translation, unknown_row], ignore_index=True)

# ---------- Missing numeric values -> median ----------
num_cols = [
    "product_name_length", "product_description_length", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
]
products[num_cols] = products[num_cols].fillna(products[num_cols].median())

print("products", products.shape, "| missing values left:", products.isnull().sum().sum())
print("category_translation", category_translation.shape)
print("Categories in products missing from translation:",
      (~products["product_category_name"].isin(category_translation["product_category_name"])).sum())

products (32951, 9) | missing values left: 0
category_translation (74, 2)
Categories in products missing from translation: 0


In [7]:
orders = orders[orders["customer_id"].isin(customers["customer_id"])]

items = items[
    items["order_id"].isin(orders["order_id"])
    & items["product_id"].isin(products["product_id"])
    & items["seller_id"].isin(sellers["seller_id"])
]

payments = payments[payments["order_id"].isin(orders["order_id"])]
reviews  = reviews[reviews["order_id"].isin(orders["order_id"])]

for name, df in {"orders": orders, "items": items, "payments": payments, "reviews": reviews}.items():
    print(name, df.shape)

orders (99441, 8)
items (112650, 7)
payments (103877, 5)
reviews (98095, 7)


In [8]:
cleaned = {
    "customers": customers,
    "geolocation": geolocation,
    "orders": orders,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}

for name, df in cleaned.items():
    df.to_csv(CLEAN / f"{name}.csv", index=False)
    print("saved", name, df.shape)

saved customers (99441, 5)
saved geolocation (19015, 5)
saved orders (99441, 8)
saved items (112650, 7)
saved payments (103877, 5)
saved reviews (98095, 7)
saved products (32951, 9)
saved sellers (3095, 4)
saved category_translation (74, 2)
